In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Đường dẫn dữ liệu
PROCESSED_DIR = '../data/processed/'

# Tải dữ liệu
df = pd.read_csv(PROCESSED_DIR + 'master_df_step1.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Tải các bảng bổ trợ để lấy đặc trưng
web_traffic = pd.read_csv(PROCESSED_DIR + 'clean_web_traffic.csv')
web_traffic['date'] = pd.to_datetime(web_traffic['date'])

promotions = pd.read_csv(PROCESSED_DIR + 'clean_promotions.csv')
promotions['start_date'] = pd.to_datetime(promotions['start_date'])
promotions['end_date'] = pd.to_datetime(promotions['end_date'])

print(f"Đã tải Master DataFrame với {df.shape[0]} dòng và {df.shape[1]} cột.")

In [ ]:
def create_time_features(df):
    df['month'] = df['Date'].dt.month
    df['year'] = df['Date'].dt.year
    df['day_of_week'] = df['Date'].dt.dayofweek # 0 là Thứ 2, 6 là Chủ nhật
    df['day_of_month'] = df['Date'].dt.day
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    df['quarter'] = df['Date'].dt.quarter
    return df

df = create_time_features(df)
print("Đã tạo các đặc trưng thời gian (Month, Day, Weekend, Quarter...).")

In [ ]:
# Lưu ý: Ta chỉ tính Lag trên cột Revenue và COGS
lags = [7, 14, 21, 28, 30] # Các mốc thời gian quan trọng (1 tuần, 2 tuần, 1 tháng)

for lag in lags:
    df[f'rev_lag_{lag}'] = df['Revenue'].shift(lag)
    df[f'cogs_lag_{lag}'] = df['COGS'].shift(lag)

# Tạo đặc trưng Trung bình trượt (Rolling Mean) trong 7 ngày và 30 ngày gần nhất
df['rev_roll_mean_7'] = df['Revenue'].shift(1).rolling(window=7).mean()
df['rev_roll_mean_30'] = df['Revenue'].shift(1).rolling(window=30).mean()

print("Đã tạo đặc trưng Lags và Rolling (7 ngày, 30 ngày).")

In [ ]:
# Nối bảng web_traffic vào df
df = pd.merge(df, web_traffic[['date', 'sessions', 'unique_visitors', 'page_views']], 
              left_on='Date', right_on='date', how='left')

# Xóa cột date thừa sau khi merge
df.drop('date', axis=1, inplace=True)

# Vì tập Test (tương lai) không có dữ liệu traffic, ta cũng phải tạo Lag cho traffic
# để mô hình dùng dữ liệu traffic của quá khứ dự báo cho tương lai
df['sessions_lag_7'] = df['sessions'].shift(7)
df['page_views_lag_7'] = df['page_views'].shift(7)

print("Đã kết hợp đặc trưng Web Traffic.")

In [ ]:
def check_promo(current_date, promo_df):
    # Kiểm tra xem ngày hiện tại có nằm giữa start_date và end_date của bất kỳ promo nào không
    is_promo = promo_df[(current_date >= promo_df['start_date']) & (current_date <= promo_df['end_date'])]
    return 1 if not is_promo.empty else 0

# Áp dụng (Lưu ý: Bước này có thể chạy hơi chậm một chút vì duyệt qua từng ngày)
print("Đang tính toán đặc trưng Khuyến mãi (vui lòng đợi)...")
df['is_promotion'] = df['Date'].apply(lambda x: check_promo(x, promotions))

print("Đã hoàn thành tạo đặc trưng Khuyến mãi!")

In [ ]:
# Xử lý các giá trị NaN sinh ra do quá trình tạo Lag (ví dụ 7 ngày đầu tiên sẽ không có lag_7)
# Ta có thể điền bằng 0 hoặc trung bình, nhưng tốt nhất là để mô hình (như LightGBM) tự xử lý
# Tuy nhiên, hãy lưu lại file sạch nhất có thể.

df.to_csv(PROCESSED_DIR + 'master_features.csv', index=False)

print("\n--- HOÀN THÀNH GIAI ĐOẠN 2 ---")
print(f"Số lượng đặc trưng hiện tại: {len(df.columns)}")
print("File đã được lưu tại: data/processed/master_features.csv")